In [13]:
from dotenv import load_dotenv
import os
from pathlib import Path
import sys
from langchain.mcp import MCPAdapter
from fastmcp.client.transports import PythonStdioTransport
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from fastmcp.mcp_config import MCPConfig
import contextlib
from fastmcp.client import Client
from fastmcp.client.group import ClientGroup

load_dotenv()

True

# MCP Adapter in LangChain

In [3]:
path_to_parent = os.path.join(Path.cwd().parent, "09_mcp")

In [ ]:
mcp_transport = PythonStdioTransport(
    script_path=Path(os.path.join(path_to_parent, "cinebot_mcp_server.py")),
    log_file=Path(os.path.join(path_to_parent, "cinebot_mcp_server.log"))
)

async with MCPAdapter(mcp_transport) as mcp_adapter:
    # Now you can use the mcp_adapter to interact with the MCP server
    tools=await mcp_adapter.list_tools()
    print("Available tools:", [t.name for t in tools])

Available tools: ['check_showtimes', 'cancel_booking', 'get_seat_map']


### Call a MCP tool 
1. Without any AI or Agent
2. With AI/Agent

In [19]:
async with MCPAdapter(mcp_transport) as adapter:
    tools = await adapter.list_tools()
    showtime_tool = next((t for t in tools if t.name == "check_showtimes"), None)
    direct_result = await showtime_tool.ainvoke({"movie_title": "interstellar"})
    print("Direct invocation result:", direct_result)

# Calling with Agent as well
    try:
        agent = create_agent(model='openai:gpt-5-nano',tools=tools)
        result = await agent.ainvoke({"messages":[('user', "What are the showtimes for Interstellar?")]})
        print("Agent invocation result:", result['messages'][-1])
    except Exception as e:
        print("Error during agent invocation:", e)

Direct invocation result: [{'type': 'text', 'text': '7:00 PM and 10:15 PM', 'id': 'lc_3bd88dcc-00bd-4974-ba46-9113ceb23bfa'}]
Agent invocation result: content='Interstellar showtimes today: 7:00 PM and 10:15 PM.\n\nWould you like me to book tickets, check seat availability, or show you the seat map? If booking, tell me how many tickets and any seating preferences.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 315, 'prompt_tokens': 236, 'total_tokens': 551, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 256, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EWSJJFE6tWr85fr7JrVBwIeZaz86L', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a11800-36e3-7211-8934-800003c

### Structured Output
* `seat_map_tool` in MCP server is returning a Json / Dict object which is a structured output. e.g.
```json
    {"movie": "movie_title", "available_rows": ["A", "B", "C"], "sold_out_rows": ["D"]}
```
* Langcahin provides 2 ways to read the Tool results returned from MCP
    * `.artifact` to access the structured data returned by the tool
    * `.content` provides the textual content that the model reads

In [ ]:
async with MCPAdapter(mcp_transport) as mcp_adapter:
    tools = await mcp_adapter.list_tools()
    seat_map_tool = next(t for t in tools if t.name == "get_seat_map")

    tool_call ={
        'name': 'get_seat_map',
        'args': {'movie_title': 'Interstellar'},
        'id': 'unique_call_id_12345',  # Optional: Provide a unique ID for the call
        'type':'tool_call'
    }

    message = await seat_map_tool.ainvoke(tool_call)
    print("Text Content:--------------------(what my model reads)\n",message.content)
    print("Dict Data: -------------------- (what my code can use if required):\n",message.artifact)
    print("Available Rows:--------------------\n", message.artifact['structured_content']['available_rows']) 

Text Content:--------------------(what my model reads)
 [{'type': 'text', 'text': '{"movie":"Interstellar","available_rows":["A","B","C"],"sold_out_rows":["D"]}', 'id': 'lc_f5c29a57-814f-4076-8a37-cd34b59b2929'}]
Dict Data: -------------------- (what my code can use if required):
 {'structured_content': {'movie': 'Interstellar', 'available_rows': ['A', 'B', 'C'], 'sold_out_rows': ['D']}}
Available Rows:--------------------
 ['A', 'B', 'C']


### Error handling
| Failure | What happened | What the agent sees |
|---|---|---|
| Server tool **ran and reported failure** (`isError=True`) | The tool executed, then explicitly failed (bad input, business-rule violation) | A `ToolMessage` with `status="error"` — **the agent can read it and self-correct**, same as any other tool error in this course |
| **Transport / session failure** | The connection dropped, the server crashed, the process couldn't start | **Raises an exception** — there is no error `ToolMessage` for the model to reason about, because nothing came back to convert |

* `error_server.py` file is a MCP server that starts and exposces the tools inside it.
* This MCP have one tool `risky_lookup` that returns an error if the booking ID format is incorrect.
* With error message "Invalid booking ID format: '<booking_id>'. Expected it to start with 'BK'."

In [ ]:
error_message_global = ''
error_transport = PythonStdioTransport(
    script_path=Path(os.path.join(path_to_parent, "cinebot_error_server.py")),
    log_file=Path(os.path.join(path_to_parent, "cinebot_error_server.log"))
)

async with MCPAdapter(error_transport) as mcp_adapter:
    tools = await mcp_adapter.list_tools()
    risky_lookup_tool = tools[0]

    bad_call = {
        'name': 'risky_lookup',
        'args': {'booking_id': '12345'}, 
        'id': 'bad_call_001', 
        'type': 'tool_call'
    }

    error_message = await risky_lookup_tool.ainvoke(bad_call)

error_message

ToolMessage(content=[{'type': 'text', 'text': "Error calling tool 'risky_lookup': Invalid booking ID format: '12345'. Expected it to start with 'BK'.", 'id': 'lc_fd55f0a0-a2b4-4f40-8ce1-4c13f8ae2868'}], name='risky_lookup', tool_call_id='bad_call_001', status='error')

### Tool metadata
* Below code lists all tools and indicates whether they are considered destructive based on their metadata.
* Requires check on `tools.metadata` for the `destructive_hint` or `read_only_hint`.
* The path to find the destructive hint in the tool metadata is:
    ```python
        tool.metadata['mcp']['tool']['annotations']['destructive_hint']
    ```
* Read optional metadata defensively, so a missing field returns a default rather than failing
* In below MCP tools only `cancel_booking` have `destructive_hint=True` as this tool will cancel ay booking which is irreversible.

In [33]:
def is_destructive_tool_call(tool) -> bool:
    """Read the MCP destructive hint off the adapter's tool metadata."""
    # Chain `.get` with defaults so a tool missing any nested field returns
    # False rather than raising.
    annotations = (
        (tool.metadata or {}).get("mcp", {}).get("tool", {}).get("annotations", {})
    )
    return annotations.get("destructive_hint", False)

async with MCPAdapter(mcp_transport) as mcp_adapter:
    tools = await mcp_adapter.list_tools()
    for t in tools:
        print(f"Tool: {t.name}, Destructive: {is_destructive_tool_call(t)}, Tools:{t}")


Tool: check_showtimes, Destructive: False, Tools:name='check_showtimes' description='Check available showtimes for a movie.' args_schema={'type': 'object', 'additionalProperties': False, 'properties': {'movie_title': {'type': 'string'}}, 'required': ['movie_title']} metadata={'mcp': {'tool': {'_meta': {'fastmcp': {'tags': []}}}, 'server': {'name': 'CineBot', 'version': '4.0.10'}}} handle_tool_error=<function _handle_mcp_tool_error at 0x000002CC8CDA6520> response_format='content_and_artifact' coroutine=<function as_langchain_tool.<locals>.call_tool at 0x000002CC8FB62980>
Tool: cancel_booking, Destructive: True, Tools:name='cancel_booking' description='Cancel an existing booking. Irreversible.' args_schema={'type': 'object', 'additionalProperties': False, 'properties': {'booking_id': {'type': 'string'}}, 'required': ['booking_id']} metadata={'mcp': {'tool': {'annotations': {'destructive_hint': True}, '_meta': {'fastmcp': {'tags': []}}}, 'server': {'name': 'CineBot', 'version': '4.0.10'}}

### Human-in-the-loop
* Based on the `meta` information provided from server's tools we can apply our custom logic.
    * Break the flow and exit.
    * Return blocking error.
    * **`Human-in-the-loop`**.
* Read the `destructive_hint` from all tools in MCP using `is_destructive_tool_call` from metadata once during tool discovery. 
* Then give the `human-in-the-loop` configuration to tools dynamically from whatever the server that are marked destructive and returns whether the call needs approval.

In [34]:
async with MCPAdapter(mcp_transport) as adapter:
    tools = await adapter.list_tools()

    for t in tools:
        print(f"{t.name}: destructive={is_destructive_tool_call(t)}")

    # One HITL config, built dynamically from whatever the server marked destructive.
    interrupt_on = {
        t.name: {"allowed_decisions": ["approve", "edit", "reject"]} for t in tools if is_destructive_tool_call(t)
    }
    print("\ninterrupt_on config:", interrupt_on)

    hitl_middleware = HumanInTheLoopMiddleware(interrupt_on=interrupt_on)
    cinebot_guarded = create_agent(model="openai:gpt-5-nano", tools=tools, middleware=[hitl_middleware])

check_showtimes: destructive=False
cancel_booking: destructive=True
get_seat_map: destructive=False

interrupt_on config: {'cancel_booking': {'allowed_decisions': ['approve', 'edit', 'reject']}}


### Elicitation: When the Server Needs to Ask a Question Mid-Call
* Some MCP tools need clarification partway through a call — "confirm you want to proceed?".
* `langchain.mcp` answers this **automatically with a LangGraph `interrupt()`**  
* Every client `MCPAdapter` builds is armed for it by default, no extra code needed to enable it.

In [ ]:
elicit_transport = PythonStdioTransport(
    script_path=Path(os.path.join(path_to_parent, "cinebot_elicit_server.py")),
    log_file=Path(os.path.join(path_to_parent, "cinebot_elicit_server.log"))
)

async with MCPAdapter(elicit_transport) as adapter:
    tools = await adapter.list_tools()
    confirm_tool = next(t for t in tools if t.name == "cancel_booking_confirm")
    tool_call = {
        "name": "cancel_booking_confirm",
        "args": {"booking_id": "BK123"},
        "id": "call_1",
        "type": "tool_call",
    }
    elicit_result = await confirm_tool.ainvoke(tool_call)
    print("status:", elicit_result.status)
    print("content:", elicit_result.content)


status: error
content: [{'type': 'text', 'text': 'elicitation via server-initiated requests is unavailable on 2026-07-28 connections.', 'id': 'lc_11a7ad02-3639-4169-9612-9d7fbaee1ab2'}]


### Connecting to Multiple Servers
* Real deployments rarely talk to just one MCP server. Two patterns, with a real tradeoff:
* To give one agent tools from several servers, choose 
    * `MCPConfig` when a single aggregate connection is enough
    * `ClientGroup` when each server needs its own connection (per-server authentication, or a shared pool configured per client).

| Pattern | What it does | Tool naming | Protocol negotiation |
|---|---|---|---|
| **`MCPConfig` dict** | One aggregate connection across several servers | Prefixed by the config key you choose | **Shared** — the whole fleet negotiates down to the oldest protocol era any member requires |
| **`ClientGroup`** | Independent connections, one per server | Namespaced `{server}_{tool}` automatically | **Independent** — each member keeps its own protocol era and auth |

**1. MCPConfig**

In [ ]:
fleet_config: MCPConfig = {
    "mcpServers": {
        "cinebot": {
            "command": sys.executable,
            "args": [str(Path(path_to_parent) / "cinebot_mcp_server.py")],
        },
        "errors": {
            "command": sys.executable,
            "args": [str(Path(path_to_parent) / "cinebot_error_server.py")],
        },
    }
}

# Jupyter's stderr stream has no Windows file descriptor (`fileno`).
# MCPConfig launches stdio servers through subprocess.Popen, so give the
# child process a real file-backed stderr stream during connection.
mcp_config_log = Path(path_to_parent) / "mcp_config.log"

In [12]:
with mcp_config_log.open("a", encoding="utf-8") as stderr:
    with contextlib.redirect_stderr(stderr):
        async with MCPAdapter(fleet_config) as adapter:
            fleet_tools = await adapter.list_tools()
print("MCPConfig fleet tools:", [t.name for t in fleet_tools])

[10/08/26 04:14:51] INFO     Proxy detected connected client - reusing existing session for all       ]8;id=16281535;file://d:\agentic_bootcamp\langchain-learning\.venv\Lib\site-packages\fastmcp\server\providers\proxy.py\proxy.py]8;;\:]8;id=16281536;file://d:\agentic_bootcamp\langchain-learning\.venv\Lib\site-packages\fastmcp\server\providers\proxy.py#1383\1383]8;;\
                             requests. This may cause context mixing in concurrent scenarios, and the              
                             session's existing settings apply, so backend results are validated                   
                             against their declared output schema rather than relayed as-is. Pass a                
                             disconnected client to avoid both.                                                    

[10/08/26 04:14:53] INFO     Proxy detected connected client - reusing existing session for all       ]8;id=16281541;file://d:\agentic_bootcamp\langchain-learning\.venv\Lib\site-packages\fastmcp\server\providers\proxy.py\proxy.py]8;;\:]8;id=16281542;file://d:\agentic_bootcamp\langchain-learning\.venv\Lib\site-packages\fastmcp\server\providers\proxy.py#1383\1383]8;;\
                             requests. This may cause context mixing in concurrent scenarios, and the              
                             session's existing settings apply, so backend results are validated                   
                             against their declared output schema rather than relayed as-is. Pass a                
                             disconnected client to avoid both.                                                    

MCPConfig fleet tools: ['cinebot_check_showtimes', 'cinebot_cancel_booking', 'cinebot_get_seat_map', 'errors_risky_lookup']


**2. ClientGroup**

In [16]:
group = ClientGroup(
    {
        "cinebot": Client(Path("cinebot_mcp_server.py")), # Local STDIO server
        "errors": Client(Path("cinebot_error_server.py"), mode = "legacy"), # Local STDIO server
        'time_track_server':Client('https://time-track-mcp-server.vercel.app/mcp/') # HTTP server
    }
)
with mcp_config_log.open("a", encoding="utf-8") as stderr:
    with contextlib.redirect_stderr(stderr):
        async with MCPAdapter(group) as adapter:
            group_tools = await adapter.list_tools()
            print("ClientGroup fleet tools:", [t.name for t in group_tools])

ClientGroup fleet tools: ['cinebot_check_showtimes', 'cinebot_cancel_booking', 'cinebot_get_seat_map', 'errors_risky_lookup', 'time_track_server_log_time', 'time_track_server_get_timesheet', 'time_track_server_get_project_summary', 'time_track_server_list_projects']


### Authentication

MCP servers that aren't your own local script usually need auth. This is configured on the
**FastMCP client**, not on `MCPAdapter` itself — `MCPAdapter` just wraps whatever client you hand
it (or builds a default one for a bare URL).

| Need | Pattern |
|---|---|
| Bearer token | `Client(url, auth=token)` |
| OAuth 2.1 (discovery, browser redirect, token exchange) | `Client(url, auth="oauth")` |
| Persisted OAuth across runs | `Client(url, auth=OAuth(mcp_url=url, token_storage=...))` |
| Different auth per server | `ClientGroup` with a different `auth=` on each member `Client` |
| Per-user auth in a deployment | A custom auth handler resolves the caller server-side; the graph factory mints/exchanges a token per user |



In [ ]:
async def load_tools_with_bearer(url: str, token: str):
    # `auth` accepts a bearer-token string, the literal "oauth", or any httpx.Auth.
    async with MCPAdapter(Client(url, auth=token)) as adapter:
        return await adapter.list_tools()

In [ ]:
# Must match AUTH_TOKEN in cinebot_mcp_server.py
CINEBOT_AUTH_TOKEN = "cinebot-secret-token"

# cinebot_mcp_server.py now serves over HTTP (port 8000) since the bearer
# token check only applies at a network boundary. Start it first with:
#   python cinebot_mcp_server.py
cinebot_tools = await load_tools_with_bearer("http://127.0.0.1:8000/mcp", CINEBOT_AUTH_TOKEN)
print("Tools over HTTP with bearer auth:", [t.name for t in cinebot_tools])